# MCDI502 · Gestión de Datos y Tecnologías Big Data con Apache Spark
## Grupo 4 

**Asignatura:** MCDI502 · Gestión de Datos y Tecnologías Big Data
**Programa:** Magíster en Ciencias de Datos e Inteligencia Artificial, Universidad Andrés Bello
**Integrantes:** Felipe Díaz Toro, Pablo Ríos Passteni, Guillermo Subiabre Chacón, Ninoska Yévenes Hernández
**Herramientas:** Apache Spark 3.5.9 (PySpark)
**Datos:** modelo histórico de los Juegos Olímpicos (deportista, equipo, evento, resultado, juego)

## 1. Configuración inicial del entorno
### 1.1 Java y Spark, y variables de entorno

In [1]:
# Instalación de Java y PySpark, y variables de entorno.
# Funciona en Google Colab y en el entorno conda del grupo (grupo4_mcdi502) en Windows, macOS o Linux.
import os
import sys

if "google.colab" in sys.modules:
    # Colab: se instala Java 17 con apt y PySpark 3.5.9 con pip
    !apt-get install openjdk-17-jdk-headless -qq > /dev/null
    %pip install -q pyspark==3.5.9 findspark
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
else:
    # Local: Java 17 viene del entorno conda (conda install -c conda-forge openjdk=17);
    # pip solo instala PySpark si falta. conda ubica Java en Library (Windows) o lib/jvm (macOS/Linux)
    %pip install -q pyspark==3.5.9 findspark
    java_dir = "Library" if os.name == "nt" else os.path.join("lib", "jvm")
    os.environ["JAVA_HOME"] = os.path.join(sys.prefix, java_dir)

# Spark debe usar el mismo Python del notebook para sus procesos de trabajo (necesario en Windows)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

assert os.path.isdir(os.environ["JAVA_HOME"]), "No se encontró Java en " + os.environ["JAVA_HOME"]
print("✅ Variables de entorno configuradas")
print("JAVA_HOME:", os.environ["JAVA_HOME"])

Note: you may need to restart the kernel to use updated packages.
✅ Variables de entorno configuradas
JAVA_HOME: C:\Users\pablo\anaconda3\envs\grupo4_mcdi502\Library


### 1.2 SparkSession y SparkContext

In [2]:
# Importar findspark para inicializar Spark
import findspark
findspark.init()

# Importar SparkSession (clase principal para trabajar con Spark)
from pyspark.sql import SparkSession

# Crear la sesión de Spark en modo local usando todos los núcleos disponibles
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Taller1 Grupo4') \
        .getOrCreate()

# Crear el SparkContext (necesario para trabajar con RDDs)
sc = spark.sparkContext

print("✅ Sesión Spark creada")
print("Spark Version:", spark.version)

✅ Sesión Spark creada
Spark Version: 3.5.9


## 2. RDDs: creación y unión (ID 3.1)
### 2.0 Ruta de los datos 


In [3]:
# El notebook está en notebooks/, por eso los datos se buscan un nivel arriba
ruta = '../data/raw/taller1_sumativa2/'

### 2.1 RDD deportista (6 particiones)

In [4]:
# Cada elemento del RDD es una línea del archivo (deportista.csv no tiene encabezado)
deportista = sc.textFile(ruta + 'deportista.csv', 6)

print("Particiones de deportista:", deportista.getNumPartitions())
deportista.take(3)

Particiones de deportista: 6


['1,A Dijiang,1,24,180,80,199',
 '2,A Lamusi,1,23,170,60,199',
 '3,Gunnar Nielsen Aaby,1,24,0,0,273']

### 2.2 RDD deportista2

In [5]:
deportista2 = sc.textFile(ruta + 'deportista2.csv')

print("Particiones de deportista2:", deportista2.getNumPartitions())
deportista2.take(3)

Particiones de deportista2: 2


['67787,Lee BongJu,1,27,167,56,970',
 '67788,Lee BuTi,1,23,164,54,203',
 '67789,Anthony N. Buddy Lee,1,34,172,62,1096']

### 2.3 Unión de RDDs: deportistaTotal

In [6]:
# union() es una transformación: junta ambos RDDs sin eliminar duplicados
deportistaTotal = deportista.union(deportista2)

print("Particiones de deportistaTotal:", deportistaTotal.getNumPartitions())

Particiones de deportistaTotal: 8


### 2.4 Cantidad de registros de deportistaTotal

In [7]:
# count() es una acción: dispara el cálculo
print("Registros en deportistaTotal:", deportistaTotal.count())

Registros en deportistaTotal: 135571


### 2.5 Conversión a DataFrame deportista

In [8]:
from pyspark.sql.functions import col

columnas = ["deportista_id", "nombre", "genero", "edad", "altura", "peso", "equipo_id"]

# split(",")[:7]: una fila de deportista.csv (id 16587) trae una coma de más y tendría 8 campos
deportista = deportistaTotal.map(lambda linea: linea.split(",")[:7]).toDF(columnas)

# Todo llega como texto: se convierte a tipo numérico (peso a double por sus decimales)
deportista = deportista.withColumn("deportista_id", col("deportista_id").cast("int")) \
                       .withColumn("genero", col("genero").cast("int")) \
                       .withColumn("edad", col("edad").cast("int")) \
                       .withColumn("altura", col("altura").cast("int")) \
                       .withColumn("peso", col("peso").cast("double")) \
                       .withColumn("equipo_id", col("equipo_id").cast("int"))

deportista.show(5)
deportista.printSchema()

+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|80.0|      199|
|            2|            A Lamusi|     1|  23|   170|60.0|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0| 0.0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0| 0.0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|82.0|      705|
+-------------+--------------------+------+----+------+----+---------+
only showing top 5 rows

root
 |-- deportista_id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- genero: integer (nullable = true)
 |-- edad: integer (nullable = true)
 |-- altura: integer (nullable = true)
 |-- peso: double (nullable = true)
 |-- equipo_id: integer (nullable = true)



## 3. RDDs: transformaciones (ID 3.2)

### 3.1 RDD MayorEdad

In [9]:
# Un deportista es mayor de edad si tiene 18 años o más
# split(",")[:7] protege de la fila corrupta (deportista_id 16587 trae una coma extra)
MayorEdad = deportistaTotal.filter(lambda linea: int(linea.split(",")[:7][3]) >= 18)

print("Deportistas mayores de edad:", MayorEdad.count())
MayorEdad.take(5)

Deportistas mayores de edad: 123244


['1,A Dijiang,1,24,180,80,199',
 '2,A Lamusi,1,23,170,60,199',
 '3,Gunnar Nielsen Aaby,1,24,0,0,273',
 '4,Edgar Lindenau Aabye,1,34,0,0,278',
 '5,Christine Jacoba Aaftink,2,21,185,82,705']

### 3.2 RDD Deportistas_mujer

In [10]:
# El campo genero/sexo: 1 = hombre, 2 = mujer
Deportistas_mujer = deportistaTotal.filter(lambda linea: linea.split(",")[:7][2] == "2")

print("Deportistas mujeres:", Deportistas_mujer.count())
Deportistas_mujer.take(5)

Deportistas mujeres: 33981


['5,Christine Jacoba Aaftink,2,21,185,82,705',
 '8,Cornelia Cor Aalten Strannood ,2,18,168,0,705',
 '13,Minna Maarit Aalto,2,30,159,55.5,350',
 '14,Pirjo Hannele Aalto Mattila ,2,32,171,65,350',
 '21,Ragnhild Margrethe Aamodt,2,27,163,0,742']

### 3.3 deportistaTotal en mayúsculas

In [11]:
# Convierte a mayúsculas cada línea completa del RDD deportistaTotal
deportistaTotalMayus = deportistaTotal.map(lambda linea: linea.upper())
deportistaTotalMayus.take(5)

['1,A DIJIANG,1,24,180,80,199',
 '2,A LAMUSI,1,23,170,60,199',
 '3,GUNNAR NIELSEN AABY,1,24,0,0,273',
 '4,EDGAR LINDENAU AABYE,1,34,0,0,278',
 '5,CHRISTINE JACOBA AAFTINK,2,21,185,82,705']

## 4. DataFrames: creación e integración (ID 3.1)

Se cargan las tablas del modelo olímpico, se optimizan las dimensiones pequeñas y se integran los datos tomando Resultado como tabla central. Se comprueban las claves y los registros sin correspondencia para explicar cualquier diferencia de filas.


### 4.1 Crear los DataFrames Equipos, Resultado y Juego

Los archivos se leen con sus separadores originales. En juegos.json, temporada contiene el año y ciudad contiene Verano o Invierno; se renombran a anio_edicion y temporada para que las agregaciones posteriores utilicen la temporada deportiva correcta. El archivo no aporta una ciudad sede en ese campo.


In [12]:
from pyspark.sql.functions import broadcast

# Cargar los archivos con sus separadores y formato correspondientes.
Equipos = spark.read.csv(ruta + 'equipo.csv', sep=',', inferSchema=True, header=True)
Resultado = spark.read.csv(ruta + 'resultados.csv', sep=';', inferSchema=True, header=True)
Juego = spark.read.option("multiline", "true").json(ruta + 'juegos.json')

# El JSON llama temporada al anio y ciudad a Verano/Invierno.
# Renombrar evita agrupar alturas por anio en lugar de por temporada.
Juego = Juego.withColumnRenamed("temporada", "anio_edicion").withColumnRenamed("ciudad", "temporada")
Juego.select("juego_id", "ano", "anio_edicion", "temporada").show(5, truncate=False)

print("Equipos:", Equipos.count(), "filas")
print("Resultado:", Resultado.count(), "filas")
print("Juego:", Juego.count(), "filas")

+--------+-----------+------------+---------+
|juego_id|ano        |anio_edicion|temporada|
+--------+-----------+------------+---------+
|1       |1896 Verano|1896        |Verano   |
|2       |1900 Verano|1900        |Verano   |
|3       |1904 Verano|1904        |Verano   |
|4       |1906 Verano|1906        |Verano   |
|5       |1908 Verano|1908        |Verano   |
+--------+-----------+------------+---------+
only showing top 5 rows



Equipos: 1184 filas
Resultado: 271116 filas


Juego: 51 filas


### 4.1.1 Crear el DataFrame Evento

Algunas filas pueden contener una capa adicional de comillas alrededor del registro completo. Se retira esa capa y se deja que el lector CSV interprete las comas del nombre del evento. Se cuentan las filas reparadas durante la ejecución para documentar el alcance de la corrección con los archivos utilizados.


In [13]:
textoEvento = sc.textFile(ruta + 'evento.csv')
headerEvento = textoEvento.first()

def limpiar_doble_comillado(linea):
    # Si la fila completa quedó envuelta en comillas por el bug de exportación,
    # se retira esa capa extra y se des-escapan las comillas dobles internas
    if linea.startswith('"') and linea.endswith('"'):
        return linea[1:-1].replace('""', '"')
    return linea

registrosEvento = textoEvento.filter(lambda l: l != headerEvento)
filas_reparadas = registrosEvento.filter(
    lambda l: l.startswith('"') and l.endswith('"')
).count()
print("Registros de eventos antes de reparar:", registrosEvento.count())
print("Registros con capa adicional de comillas:", filas_reparadas)
filasEvento = registrosEvento.map(limpiar_doble_comillado)
Evento = spark.read.csv(filasEvento, sep=',', inferSchema=True, header=False).toDF(*headerEvento.split(','))

print("Evento:", Evento.count(), "filas")
Evento.filter(col("evento").contains(",")).show(5, truncate=False)

Registros de eventos antes de reparar: 765
Registros con capa adicional de comillas: 280


Evento: 765 filas


+----------+-------------------------------------------------+----------+
|cod_evento|evento                                           |deporte_id|
+----------+-------------------------------------------------+----------+
|30        |Art Competitions Mixed Sculpturing, Unknown Event|14        |
|38        |Wrestling Men's Light-Heavyweight, Greco-Roman   |18        |
|40        |Speed Skating Men's 1,500 metres                 |5         |
|41        |Gymnastics Men's Team All-Around, Free System    |13        |
|50        |Athletics Men's 1,500 metres                     |7         |
+----------+-------------------------------------------------+----------+
only showing top 5 rows



### 4.2 Optimización de los DataFrames

Evento, Equipos y Juego son dimensiones pequeñas respecto de Resultado. Se aplica cache para reutilizar sus datos; el almacenamiento se materializa cuando una acción los evalúa. Los joins usan broadcast para transmitir estas dimensiones a los ejecutores y evitar redistribuir la tabla grande por esas uniones. La unión con deportista puede requerir shuffle. Estas decisiones no implican una mejora de tiempo medida en este notebook.


In [14]:
# cache es perezoso: count materializa las dimensiones para su reutilizacion.
for nombre_tabla, tabla in [("Evento", Evento), ("Equipos", Equipos), ("Juego", Juego)]:
    tabla.cache()
    print(f"{nombre_tabla}: filas={tabla.count()}, cache={tabla.is_cached}")


Evento: filas=765, cache=True
Equipos: filas=1184, cache=True


Juego: filas=51, cache=True


### 4.3 DataFrame integrado (deportista + evento + equipo + resultado + juego)

`Resultado` es la tabla central (conecta deportista, juego y evento). Se arma paso a paso para evitar columnas ambiguas en los joins.

In [15]:
paso1 = Resultado.join(broadcast(Juego), "juego_id")
paso2 = paso1.join(broadcast(Evento), paso1["evento_id"] == Evento["cod_evento"])
paso3 = paso2.join(deportista, "deportista_id")
integrado = paso3.join(broadcast(Equipos), paso3["equipo_id"] == Equipos["id"])

print("Resultado original:", Resultado.count(), "filas")
print("DataFrame integrado:", integrado.count(), "filas")
print("La diferencia se analiza por cada unión en la sección 4.4.")

integrado.select("deportista_id", "nombre", "medalla", "evento", "equipo", "ano").show(5, truncate=False)
# El plan permite inspeccionar las estrategias elegidas por Spark para los joins.
integrado.explain()


Resultado original: 271116 filas


DataFrame integrado: 270907 filas
La diferencia se analiza por cada unión en la sección 4.4.


+-------------+--------------------+-------+-------------------------------------+------+-------------+
|deportista_id|nombre              |medalla|evento                               |equipo|ano          |
+-------------+--------------------+-------+-------------------------------------+------+-------------+
|148          |Kathryn Katie Abbott|NA     |Sailing Women's Three Person Keelboat|Canada|2008 Verano  |
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Super G        |Latvia|2014 Invierno|
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Giant Slalom   |Latvia|2014 Invierno|
|463          |Agnese boltia       |NA     |Alpine Skiing Women's Slalom         |Latvia|2014 Invierno|
|471          |Ashraf Abou ElHassan|NA     |Volleyball Men's Volleyball          |Egypt |2000 Verano  |
+-------------+--------------------+-------+-------------------------------------+------+-------------+
only showing top 5 rows

== Physical Plan ==
AdaptiveSparkPlan i

### 4.4 Validación de las uniones

Se verifican claves únicas en las tablas que se incorporan para evitar multiplicar resultados. Cada left_anti identifica las filas que quedan fuera de su correspondiente inner join. Las pérdidas se miden por etapa, por lo que pueden sumarse sin contar una misma fila dos veces. Los valores sin correspondencia se muestran para comprobar su causa, sin presuponer que todos son #N/A. Se conserva el criterio de incluir solo resultados con correspondencia en las cinco tablas.


In [16]:
# Comprobar que cada clave de las dimensiones identifica una sola fila.
for tabla, clave, nombre_tabla in [
    (Juego, "juego_id", "Juego"),
    (Evento, "cod_evento", "Evento"),
    (deportista, "deportista_id", "deportista"),
    (Equipos, "id", "Equipos"),
]:
    duplicadas = tabla.groupBy(clave).count().filter(col("count") > 1)
    if duplicadas.take(1):
        duplicadas.show(10, truncate=False)
        raise ValueError(f"Claves duplicadas en {nombre_tabla}: revisar antes de agregar resultados")
    print(f"{nombre_tabla}: claves únicas verificadas")

etapas = [
    ("Juego", Resultado, Juego, "juego_id", "juego_id", paso1),
    ("Evento", paso1, Evento, "evento_id", "cod_evento", paso2),
    ("deportista", paso2, deportista, "deportista_id", "deportista_id", paso3),
    ("Equipos", paso3, Equipos, "equipo_id", "id", integrado),
]
perdidas = 0
for nombre_tabla, entrada, dimension, clave_entrada, clave_dimension, salida in etapas:
    sin_correspondencia = entrada.join(
        dimension, entrada[clave_entrada] == dimension[clave_dimension], "left_anti"
    )
    n_entrada = entrada.count()
    n_salida = salida.count()
    n_excluidas = sin_correspondencia.count()
    print(f"{nombre_tabla}: entrada={n_entrada}, salida={n_salida}, sin correspondencia={n_excluidas}")
    sin_correspondencia.groupBy(clave_entrada).count().orderBy(col("count").desc()).show(10, truncate=False)
    assert n_entrada == n_salida + n_excluidas, f"Conteo inconsistente al unir {nombre_tabla}"
    perdidas += n_excluidas

assert Resultado.count() - integrado.count() == perdidas
print("Total de resultados excluidos por falta de correspondencia:", perdidas)


Juego: claves únicas verificadas


Evento: claves únicas verificadas


deportista: claves únicas verificadas


Equipos: claves únicas verificadas


Juego: entrada=271116, salida=271116, sin correspondencia=0


+--------+-----+
|juego_id|count|
+--------+-----+
+--------+-----+



... (3 outputs left out)

+-------------+-----+
|deportista_id|count|
+-------------+-----+
+-------------+-----+



Equipos: entrada=271107, salida=270907, sin correspondencia=200


+---------+-----+
|equipo_id|count|
+---------+-----+
|0        |200  |
+---------+-----+



Total de resultados excluidos por falta de correspondencia: 209


## 5. Paralelismo (ID 3.4)

Se redistribuye el DataFrame integrado con repartition(5). Esta transformación implica shuffle y fija cinco particiones para el procesamiento posterior. Se muestra el número antes y después y se comprueba que la operación conserva las filas. Cinco particiones responde al requisito del taller; no se presenta como un valor óptimo medido.

El resultado se persiste con MEMORY_AND_DISK porque se reutiliza en conteos, inspección, columnas calculadas y agregaciones. El primer count materializa sus datos; si no caben en memoria, pueden almacenarse en disco. Al terminar el análisis se deben liberar los datos persistidos con unpersist().


In [17]:
from pyspark import StorageLevel

filas_antes = integrado.count()
print("Particiones antes:", integrado.rdd.getNumPartitions())

# Conservar esta referencia para liberar el cache al finalizar todo el notebook.
integrado_5_particiones = integrado.repartition(5).persist(StorageLevel.MEMORY_AND_DISK)
filas_despues = integrado_5_particiones.count()
particiones_despues = integrado_5_particiones.rdd.getNumPartitions()
integrado = integrado_5_particiones

print("Particiones después:", particiones_despues)
print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Nivel de persistencia:", integrado.storageLevel)
assert particiones_despues == 5, "El DataFrame debe tener cinco particiones"
assert filas_antes == filas_despues, "El reparticionamiento debe conservar las filas"

# Ejecutar al terminar todas las secciones, no antes de reutilizar los datos:
# integrado_5_particiones.unpersist()
# for tabla in (Evento, Equipos, Juego):
#     tabla.unpersist()


Particiones antes: 20


Particiones después: 5
Filas antes: 270907
Filas después: 270907
Nivel de persistencia: Disk Memory Serialized 1x Replicated


## 6. Inspección del DataFrame (ID 3.1)

En este punto se revisa la estructura del DataFrame `integrado` antes de crear columnas calculadas y agregaciones. Se muestran los tres elementos solicitados:

- **Cantidad de filas** (`count`): resultados con correspondencia en las cinco tablas.
- **Tipo de datos** (`dtypes`): el tipo de cada columna, para confirmar que las columnas numéricas pueden usarse en cálculos.
- **Esquema** (`printSchema`): la estructura completa, que además indica si cada columna admite nulos.

También se muestran la cantidad de columnas y de particiones, para confirmar que se trabaja con el DataFrame de cinco particiones de la Sección 5. Las columnas `evento_id`/`cod_evento` y `equipo_id`/`id` aparecen por separado porque son las claves de los joins. `evento_id` queda como texto porque resultados.csv contiene valores `#N/A`.

In [18]:
# Inspección del DataFrame integrado (5 particiones, persistido en la Sección 5)
print("Cantidad de filas:", integrado.count())
print("Cantidad de columnas:", len(integrado.columns))
print("Particiones:", integrado.rdd.getNumPartitions())

# Tipo de datos de cada columna
print("\nTipos de datos:")
for nombre, tipo in integrado.dtypes:
    print(f"  {nombre:<15} {tipo}")

# Esquema completo (nombre, tipo y si acepta nulos)
print("\nEsquema:")
integrado.printSchema()

Cantidad de filas: 270907
Cantidad de columnas: 20
Particiones: 5

Tipos de datos:
  deportista_id   int
  juego_id        int
  resultado_id    int
  medalla         string
  evento_id       string
  ano             string
  temporada       string
  anio_edicion    bigint
  cod_evento      int
  evento          string
  deporte_id      string
  nombre          string
  genero          int
  edad            int
  altura          int
  peso            double
  equipo_id       int
  id              int
  equipo          string
  sigla           string

Esquema:
root
 |-- deportista_id: integer (nullable = true)
 |-- juego_id: integer (nullable = true)
 |-- resultado_id: integer (nullable = true)
 |-- medalla: string (nullable = true)
 |-- evento_id: string (nullable = true)
 |-- ano: string (nullable = true)
 |-- temporada: string (nullable = true)
 |-- anio_edicion: long (nullable = true)
 |-- cod_evento: integer (nullable = true)
 |-- evento: string (nullable = true)
 |-- deporte_id: s

## 7. Columnas calculadas

Se calculan sobre el DataFrame `integrado` de la Sección 4 (no sobre `deportista` solo), tal como pide la rúbrica: *"se observan las columnas en salidas del DataFrame integrado"*.

**Hallazgo de calidad de datos:** el 23.7% de las filas de `integrado` corresponde a deportistas con `altura=0` y/o `peso=0` (dato faltante en el dataset original). El `IMC` se deja en `NULL` para esas filas en vez de inventar un valor — se documenta el porcentaje en la Sección 7.1.

### 7.1 Columna calculada IMC

In [19]:
from pyspark.sql.functions import col, round, when

# El 23.7% de las filas de integrado tiene altura=0 y/o peso=0 en el dataset original (dato
# faltante del censo histórico, no un error de carga). Se mide para dejarlo documentado.
total = integrado.count()
faltantes = integrado.filter((col("altura") == 0) | (col("peso") == 0)).count()
print(f"Filas con altura o peso faltante (=0): {faltantes} de {total} ({faltantes/total:.1%})")

# Se tratan explícitamente como NULL antes de dividir (en vez de depender de que 0/0
# lo haga implícito): así el IMC queda NULL solo cuando el dato de origen falta.
# IMC = peso (kg) / altura^2 (m); la altura del dataset está en cm, se convierte a metros
altura_m = when(col("altura") == 0, None).otherwise(col("altura") / 100)
peso_valido = when(col("peso") == 0, None).otherwise(col("peso"))

integrado = integrado.withColumn("IMC", round(peso_valido / altura_m ** 2, 2))

integrado.select("nombre", "altura", "peso", "IMC").show(5)

Filas con altura o peso faltante (=0): 64196 de 270907 (23.7%)
+--------------------+------+----+-----+
|              nombre|altura|peso|  IMC|
+--------------------+------+----+-----+
|       Anna Janoukov|     0| 0.0| NULL|
|Bohdan Sergiyevyc...|   188|85.0|24.05|
|     Delmo Delmastro|   168|70.0| 24.8|
|John Luke Vaughan...|   186|84.0|24.28|
| Shigeki Wakabayashi|   177|77.0|24.58|
+--------------------+------+----+-----+
only showing top 5 rows



### 7.2 Columna calculada Descripcion_sexo

In [20]:
from pyspark.sql.functions import when

# sexo=1 es hombre, sexo=2 es mujer
integrado = integrado.withColumn(
    "Descripcion_sexo",
    when(col("genero") == 1, "Hombre").when(col("genero") == 2, "Mujer").otherwise("Desconocido")
)

integrado.select("nombre", "genero", "Descripcion_sexo").show(5)

+--------------------+------+----------------+
|              nombre|genero|Descripcion_sexo|
+--------------------+------+----------------+
|       Anna Janoukov|     2|           Mujer|
|Bohdan Sergiyevyc...|     1|          Hombre|
|     Delmo Delmastro|     1|          Hombre|
|John Luke Vaughan...|     1|          Hombre|
| Shigeki Wakabayashi|     1|          Hombre|
+--------------------+------+----------------+
only showing top 5 rows



## 8. Agregaciones requeridas (ID 3.3)

Se generan las cuatro agregaciones del taller sobre el DataFrame `integrado`, alternando la API de DataFrames (`groupBy`, `agg`, `pivot`) y Spark SQL (`createOrReplaceTempView` + `spark.sql`). Ambas formas producen DataFrames y las optimiza el mismo motor de Spark (Catalyst).

Criterios aplicados en toda la sección:

- **Unidad de conteo: participaciones.** Cada fila de `integrado` es un resultado: un deportista en un evento de una edición de los Juegos. Un deportista que participó en varios eventos aporta una fila por cada participación.
- **Medallas registradas por deportista.** En los deportes de equipo cada integrante recibe su medalla, por lo que una medalla de equipo cuenta una vez por integrante, tal como viene en resultados.csv.
- **Ceros como dato faltante.** `edad = 0` y `altura = 0` se excluyen de los cálculos (8.0), con el mismo criterio que el IMC de la Sección 7.

### 8.0 Preparación: los ceros son datos faltantes

En el dataset, `edad = 0` y `altura = 0` indican que el dato no se registró: no existen deportistas de 0 años ni de 0 cm. Si se incluyeran, los mínimos serían 0 y los promedios bajarían artificialmente; por ejemplo, la altura promedio de Invierno caería de 174,6 a 144,7 cm. Por eso se crean dos columnas, `edad_valida` y `altura_valida`, que dejan esos valores en NULL. Las funciones de agregación (SUM, AVG, MAX, MIN) ignoran los NULL. Es el mismo criterio que se aplicó al IMC en la Sección 7.

Luego se registra el DataFrame como vista temporal `olimpicos`, para consultarlo con Spark SQL en los puntos 8.2 y 8.4.

In [21]:
import pyspark.sql.functions as F

# Se usa el alias F para no confundir sum, max, min y round de Spark con las funciones de Python
# edad = 0 y altura = 0 son datos no registrados: se convierten a NULL (las agregaciones ignoran NULL)
integrado = integrado.withColumn("edad_valida", F.when(F.col("edad") > 0, F.col("edad"))) \
                     .withColumn("altura_valida", F.when(F.col("altura") > 0, F.col("altura")))

# Evidencia: cuántas filas quedan fuera de cada cálculo por dato faltante
integrado.select(
    F.count("*").alias("filas"),
    F.sum(F.col("edad_valida").isNull().cast("int")).alias("edad_faltante"),
    F.sum(F.col("altura_valida").isNull().cast("int")).alias("altura_faltante"),
).show()

# Vista temporal para consultar el DataFrame integrado con Spark SQL
integrado.createOrReplaceTempView("olimpicos")
print("Vista temporal 'olimpicos' registrada para Spark SQL")

+------+-------------+---------------+
| filas|edad_faltante|altura_faltante|
+------+-------------+---------------+
|270907|         9455|          60113|
+------+-------------+---------------+

Vista temporal 'olimpicos' registrada para Spark SQL


### 8.1 Medallas por equipo (API de DataFrames, con pivot)

Se cuenta cuántas medallas de oro, plata y bronce obtuvo cada equipo. Se descartan las participaciones sin medalla (`medalla = 'NA'`), se agrupa por equipo y sigla, y `pivot` convierte cada tipo de medalla en una columna. Si un equipo no ganó algún tipo de medalla, ese valor queda en 0 (`na.fill(0)`). Se agrega el total y se ordena como el medallero olímpico: primero por oros, luego por platas y bronces. La agrupación es por equipo, como pide el enunciado; los equipos con sufijo (por ejemplo, `Australia-1`) están registrados como equipos distintos en el dataset.

In [22]:
# Cantidad de medallas de oro, plata y bronce por equipo (API de DataFrames)
medallas_equipo = (integrado
    .filter(F.col("medalla") != "NA")                      # solo participaciones con medalla
    .groupBy("equipo", "sigla")
    .pivot("medalla", ["Gold", "Silver", "Bronze"])       # una columna por tipo de medalla
    .count()
    .na.fill(0)                                            # 0 si el equipo no ganó ese tipo de medalla
    .withColumnRenamed("Gold", "Oro")
    .withColumnRenamed("Silver", "Plata")
    .withColumnRenamed("Bronze", "Bronce")
    .withColumn("Total", F.col("Oro") + F.col("Plata") + F.col("Bronce"))
    .orderBy(F.desc("Oro"), F.desc("Plata"), F.desc("Bronce")))

print("Equipos con al menos una medalla:", medallas_equipo.count())
medallas_equipo.show(10, truncate=False)

Equipos con al menos una medalla: 502


+-------------+-----+----+-----+------+-----+
|equipo       |sigla|Oro |Plata|Bronce|Total|
+-------------+-----+----+-----+------+-----+
|United States|USA  |2515|1546 |1249  |5310 |
|Soviet Union |URS  |1110|766  |728   |2604 |
|Germany      |GER  |636 |603  |651   |1890 |
|Italy        |ITA  |536 |507  |487   |1530 |
|Great Britain|GBR  |531 |591  |578   |1700 |
|France       |FRA  |452 |524  |582   |1558 |
|Sweden       |SWE  |450 |479  |510   |1439 |
|Hungary      |HUN  |429 |329  |363   |1121 |
|Canada       |CAN  |422 |407  |407   |1236 |
|East Germany |GDR  |409 |325  |283   |1017 |
+-------------+-----+----+-----+------+-----+
only showing top 10 rows



### 8.2 Edad por tipo de medalla (Spark SQL)

Se calcula la suma, el promedio, el máximo y el mínimo de la edad de los deportistas según el tipo de medalla obtenida, con una consulta SQL sobre la vista `olimpicos`. `CASE` traduce las medallas al español e incluye el grupo **Sin medalla** (`medalla = 'NA'`) como referencia, para comparar a los medallistas con el resto de los participantes. Los grupos se ordenan como en el medallero: oro, plata, bronce y sin medalla. Se usa `edad_valida`, así que las edades no registradas no se consideran.

In [23]:
# Suma, promedio, máximo y mínimo de la edad por tipo de medalla (Spark SQL)
# Se incluye "Sin medalla" (medalla = 'NA') como grupo de referencia
edad_medalla = spark.sql("""
    SELECT CASE medalla WHEN 'Gold'   THEN 'Oro'
                        WHEN 'Silver' THEN 'Plata'
                        WHEN 'Bronze' THEN 'Bronce'
                        ELSE 'Sin medalla' END AS tipo_medalla,
           SUM(edad_valida)           AS suma_edad,
           ROUND(AVG(edad_valida), 2) AS promedio_edad,
           MAX(edad_valida)           AS max_edad,
           MIN(edad_valida)           AS min_edad
    FROM olimpicos
    GROUP BY medalla
    ORDER BY CASE medalla WHEN 'Gold' THEN 1 WHEN 'Silver' THEN 2 WHEN 'Bronze' THEN 3 ELSE 4 END
""")
edad_medalla.show()

+------------+---------+-------------+--------+--------+
|tipo_medalla|suma_edad|promedio_edad|max_edad|min_edad|
+------------+---------+-------------+--------+--------+
|         Oro|   313286|         23.7|      64|      11|
|       Plata|   304474|        23.75|      73|      11|
|      Bronce|   308042|        23.69|      72|      10|
| Sin medalla|  5271221|         23.7|      97|      11|
+------------+---------+-------------+--------+--------+



### 8.3 DataFrame temporada: altura por temporada (API de DataFrames)

Se crea el DataFrame `temporada` con la suma, el promedio, el máximo y el mínimo de la altura (en cm) de los deportistas que compitieron en los Juegos de Verano y de Invierno. La columna `temporada` proviene de juegos.json y se renombró en la Sección 4.1, porque en el archivo original el campo `temporada` contiene el año. Se agrupa con `groupBy` + `agg` y se usa `altura_valida`, así que las alturas no registradas no se consideran.

In [24]:
# DataFrame temporada: suma, promedio, máximo y mínimo de la altura (cm) por temporada (API de DataFrames)
temporada = integrado.groupBy("temporada").agg(
    F.sum("altura_valida").alias("suma_altura"),
    F.round(F.avg("altura_valida"), 2).alias("promedio_altura"),
    F.max("altura_valida").alias("max_altura"),
    F.min("altura_valida").alias("min_altura"),
).orderBy("temporada")

temporada.show()

+---------+-----------+---------------+----------+----------+
|temporada|suma_altura|promedio_altura|max_altura|min_altura|
+---------+-----------+---------------+----------+----------+
| Invierno|    7027252|         174.59|       211|       137|
|   Verano|   29933222|         175.52|       226|       127|
+---------+-----------+---------------+----------+----------+



### 8.4 DataFrame sexo: edad por sexo (Spark SQL)

Se crea el DataFrame `sexo` con la suma, el promedio, el máximo y el mínimo de la edad de los deportistas, agrupando por la columna `Descripcion_sexo` creada en la Sección 7.2 (1 = Hombre, 2 = Mujer). La consulta sigue la forma vista en clase (`SELECT ... GROUP BY` sobre una vista temporal) y `spark.sql` devuelve un DataFrame, igual que la API. Se usa `edad_valida`, así que las edades no registradas no se consideran.

In [25]:
# DataFrame sexo: suma, promedio, máximo y mínimo de la edad por sexo (Spark SQL)
sexo = spark.sql("""
    SELECT Descripcion_sexo           AS sexo,
           SUM(edad_valida)           AS suma_edad,
           ROUND(AVG(edad_valida), 2) AS promedio_edad,
           MAX(edad_valida)           AS max_edad,
           MIN(edad_valida)           AS min_edad
    FROM olimpicos
    GROUP BY Descripcion_sexo
    ORDER BY sexo
""")
sexo.show()

+------+---------+-------------+--------+--------+
|  sexo|suma_edad|promedio_edad|max_edad|min_edad|
+------+---------+-------------+--------+--------+
|Hombre|  4578506|        24.43|      97|      10|
| Mujer|  1618517|        21.86|      74|      11|
+------+---------+-------------+--------+--------+



### 8.5 Liberar memoria

Terminado el análisis, se liberan los datos que quedaron almacenados: el DataFrame de cinco particiones persistido con MEMORY_AND_DISK (Sección 5) y las dimensiones Evento, Equipos y Juego en caché (Sección 4.2). `unpersist()` devuelve esa memoria y ese disco a Spark para otros trabajos. También se elimina la vista temporal `olimpicos`. Al final, `is_cached` confirma que ningún DataFrame sigue almacenado.

In [26]:
# Liberar los datos persistidos (Sección 5) y las dimensiones en caché (Sección 4.2)
integrado_5_particiones.unpersist()
for tabla in (Evento, Equipos, Juego):
    tabla.unpersist()

# Eliminar la vista temporal usada por Spark SQL
spark.catalog.dropTempView("olimpicos")

for nombre_tabla, tabla in [("integrado (5 particiones)", integrado_5_particiones),
                            ("Evento", Evento), ("Equipos", Equipos), ("Juego", Juego)]:
    print(f"{nombre_tabla}: cache={tabla.is_cached}")

integrado (5 particiones): cache=False
Evento: cache=False
Equipos: cache=False
Juego: cache=False
